# NYC Taxi Demand & Fare Intelligence

## 01 — Data Exploration & Quality

### Objective

This notebook investigates and evaluates the raw NYC Yellow Taxi trip records used for the project.

The analysis focuses on understanding:

- the structure and meaning of the dataset
- the characteristics of individual taxi trips
- temporal and geographic information
- missing and anomalous values
- potential data-quality issues
- variables relevant to demand forecasting and fare prediction

The goal of this stage is to understand the raw data before performing feature engineering or machine learning.

### Data Source

NYC Taxi & Limousine Commission (TLC) Trip Record Data.

### Study Period

January to March 2026

### Downstream Tasks

The cleaned dataset will subsequently be used for:

1. Taxi demand analysis and forecasting
2. Taxi fare analysis and prediction 

In [16]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
import duckdb 

from pathlib import Path
import pyarrow.parquet as pq

pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",100)

sns.set_theme(style="whitegrid")

In [17]:
PROJECT_ROOT = Path("..")
RAW_DATA_DIR = PROJECT_ROOT/"data"/"raw"
PROCESSED_DATA_DIR = PROJECT_ROOT/"data"/"processed"
RESULTS_DIR = PROJECT_ROOT/"results"

print("Raw data directory:" , RAW_DATA_DIR)
print("Processed data directory:" , PROCESSED_DATA_DIR)
print("Results directory:" , RESULTS_DIR)

Raw data directory: ..\data\raw
Processed data directory: ..\data\processed
Results directory: ..\results


In [18]:
files = sorted(RAW_DATA_DIR.glob("*parquet"))
for file in files:
    print(file.name)

yellow_tripdata_2026-01.parquet
yellow_tripdata_2026-02.parquet
yellow_tripdata_2026-03.parquet


In [19]:
for file in files:
    size_gb = file.stat().st_size/(1024**3)
    print(f"{file.name}:{size_gb:.2f} GB")

yellow_tripdata_2026-01.parquet:0.06 GB
yellow_tripdata_2026-02.parquet:0.05 GB
yellow_tripdata_2026-03.parquet:0.06 GB


In [20]:
file_path = files[0]
print(file_path)

..\data\raw\yellow_tripdata_2026-01.parquet


In [21]:
schema = duckdb.sql(f"""DESCRIBE SELECT * FROM '{file_path}'""")
schema

┌───────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│      column_name      │ column_type │  null   │   key   │ default │  extra  │
│        varchar        │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ VendorID              │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_pickup_datetime  │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ tpep_dropoff_datetime │ TIMESTAMP   │ YES     │ NULL    │ NULL    │ NULL    │
│ passenger_count       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_distance         │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ RatecodeID            │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ store_and_fwd_flag    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ PULocationID          │ INTEGER     │ YES     │ NULL    │ NULL    │ NULL    │
│ DOLocationID          │ INTEGER     │ 

## 2 Sample Records

Before performing any transformations, we inspect a small sample of the raw trip records to understand the values represented by each field.

In [22]:
sample = duckdb.sql(f"""SELECT * FROM '{file_path}' LIMIT 10""").df()
sample

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,1,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,1,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,1,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,1,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,1,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
5,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2,2.33,1,N,144,137,1,14.2,1.00,0.5,4.99,0.0,1.0,24.94,2.5,0.0,0.75
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1,1.30,1,N,142,50,2,11.4,4.25,0.5,0.00,0.0,1.0,17.15,2.5,0.0,0.75
7,1,2026-01-01 00:34:28,2026-01-01 00:59:05,0,2.90,1,N,50,234,1,22.6,4.25,0.5,5.65,0.0,1.0,34.00,2.5,0.0,0.75
8,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1,5.34,1,N,161,45,1,37.3,1.00,0.5,8.61,0.0,1.0,51.66,2.5,0.0,0.75
9,2,2026-01-01 00:41:07,2026-01-01 00:50:42,3,1.83,1,N,237,263,1,10.7,1.00,0.5,2.36,0.0,1.0,18.06,2.5,0.0,0.00


In [23]:
duckdb.sql(f"""SELECT tpep_pickup_datetime,  tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        PULocationID,
        DOLocationID,
        fare_amount,
        tip_amount,
        tolls_amount,
        total_amount,
        cbd_congestion_fee
    FROM '{file_path}'
    LIMIT 10
""").df()

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,tip_amount,tolls_amount,total_amount,cbd_congestion_fee
0,2026-01-01 00:54:04,2026-01-01 00:59:37,1,0.97,239,238,7.2,3.66,0.0,15.86,0.00
1,2026-01-01 00:34:04,2026-01-01 00:39:47,0,0.90,163,162,7.9,0.00,0.0,13.65,0.75
2,2026-01-01 00:57:06,2026-01-01 01:05:59,0,1.40,43,237,10.7,2.50,0.0,18.95,0.75
3,2026-01-01 00:15:22,2026-01-01 00:58:10,4,5.58,142,209,38.7,11.11,0.0,55.56,0.75
4,2026-01-01 00:27:13,2026-01-01 00:40:43,0,2.16,88,144,13.5,3.85,0.0,23.10,0.75
5,2026-01-01 00:47:11,2026-01-01 01:00:47,2,2.33,144,137,14.2,4.99,0.0,24.94,0.75
6,2026-01-01 00:17:54,2026-01-01 00:28:32,1,1.30,142,50,11.4,0.00,0.0,17.15,0.75
7,2026-01-01 00:34:28,2026-01-01 00:59:05,0,2.90,50,234,22.6,5.65,0.0,34.00,0.75
8,2026-01-01 00:34:14,2026-01-01 01:11:58,1,5.34,161,45,37.3,8.61,0.0,51.66,0.75
9,2026-01-01 00:41:07,2026-01-01 00:50:42,3,1.83,237,263,10.7,2.36,0.0,18.06,0.00


In [24]:
for file in files:
    parquet_file = pq.ParquetFile(file)
    rows = parquet_file.metadata.num_rows

    print(f"{file.name} : {rows:,} rows")

yellow_tripdata_2026-01.parquet : 3,724,889 rows
yellow_tripdata_2026-02.parquet : 3,399,866 rows
yellow_tripdata_2026-03.parquet : 3,952,451 rows


## 3. Data Quality Assessment

Before modeling, we assess the completeness, validity, consistency, and plausibility of the raw trip records.

The audit focuses on:

- missing values
- duplicate records
- temporal validity
- invalid trip durations
- invalid distances
- invalid fares
- unusual passenger counts
- categorical value consistency
- extreme values and outliers

In [25]:
quality_query = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) - COUNT(tpep_pickup_datetime) AS missing_pickup,
    COUNT(*) - COUNT(tpep_dropoff_datetime) AS missing_dropoff,
    COUNT(*) - COUNT(passenger_count) AS missing_passenger_count,
    COUNT(*) - COUNT(trip_distance) AS missing_trip_distance,
    COUNT(*) - COUNT(fare_amount) AS missing_fare,

    MIN(tpep_pickup_datetime) AS earliest_pickup,
    MAX(tpep_pickup_datetime) AS latest_pickup,

    MIN(trip_distance) AS min_distance,
    MAX(trip_distance) AS max_distance,

    MIN(fare_amount) AS min_fare,
    MAX(fare_amount) AS max_fare,

    MIN(passenger_count) AS min_passengers,
    MAX(passenger_count) AS max_passengers

FROM '{files[0]}'
"""

quality_jan = duckdb.sql(quality_query).df()

quality_jan.T

,0
total_rows,3724889
missing_pickup,0
missing_dropoff,0
missing_passenger_count,1088058
missing_trip_distance,0
missing_fare,0
earliest_pickup,2025-12-31 23:57:29
latest_pickup,2026-02-01 00:45:01
min_distance,0.0
max_distance,269097.48


In [26]:
zero_passenger = duckdb.sql(f"""SELECT COUNT(*) AS zero_passenger_trips FROM '{files[0]}' WHERE passenger_count = 0""").fetchone()[0]
print(f"zero-passenger trips : {zero_passenger:,}")

zero-passenger trips : 14,787


In [27]:
total_jan = quality_jan.loc[0,"total_rows"]
print(f"percentage : {zero_passenger/total_jan * 100:.2f}%")

percentage : 0.40%


In [28]:
duration_stats = duckdb.sql(f"""
SELECT
    MIN(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime))
    ) AS min_duration_seconds,

    MAX(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime))
    ) AS max_duration_seconds,

    AVG(
        EXTRACT(EPOCH FROM (tpep_dropoff_datetime - tpep_pickup_datetime))
    ) AS avg_duration_seconds,

    COUNT(*) FILTER (
        WHERE tpep_dropoff_datetime < tpep_pickup_datetime
    ) AS negative_duration_trips,

    COUNT(*) FILTER (
        WHERE tpep_dropoff_datetime = tpep_pickup_datetime
    ) AS zero_duration_trips

FROM '{files[0]}'
""").df()

duration_stats.T

,0
min_duration_seconds,-702.00000
max_duration_seconds,450474.00000
avg_duration_seconds,1031.58962
negative_duration_trips,1.00000
zero_duration_trips,45069.00000


### Initial Data Quality Findings

The initial audit of the January 2026 Yellow Taxi records reveals several data-quality considerations:

- The January file contains a small number of records outside the calendar month, indicating that study-period filtering should be based on trip timestamps rather than filenames.
- `passenger_count` is missing for a substantial proportion of records and should not be treated as equivalent to zero.
- `trip_distance` contains extreme values that require investigation before modeling.
- `fare_amount` contains negative and unusually large values that require investigation.
- Trip durations include zero-duration, negative-duration, and extremely long records.
- Passenger counts range from 0 to 9 among the observed records.

These findings will be investigated further before any cleaning rules are established.